In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('/content/cgpa.txt')

In [ ]:
df.head()

,cgpa,resume_score,package
0,6.5,6.8,5.2
1,7.0,7.2,6.0
2,8.2,8.0,8.5
3,5.8,6.0,4.5
4,9.0,8.8,10.5


# Back probagation for Regression

In [ ]:
import numpy as np

class Backprobagation:
    def __init__(self, epochs=100, lr=0.001):
        self.epochs = epochs
        self.lr = lr
        self.weighted_sum_dict = {}
        self.weights = {}
        self.bias = {}
        # initialization stage
        self.weights[0] = np.ones((2, 2))
        self.bias[0] = np.ones(2,)
        self.weights[1] = np.ones((2, 1))
        self.bias[1] = np.ones(1,)

    def forward_probagation(self, X_train):
        # forward propagation
        self.weighted_sum_dict[0] = np.dot(X_train, self.weights[0]) + self.bias[0]
        self.weighted_sum_dict[1] = np.dot(self.weighted_sum_dict[0], self.weights[1]) + self.bias[1]
        y_pred = self.weighted_sum_dict[1]
        return y_pred

    def fit(self, X_train, y_train):
        for i in range(self.epochs):
            y_pred = self.forward_probagation(X_train)
            loss = np.mean((y_train - y_pred) ** 2)
            dz1 = np.dot((y_pred - y_train), self.weights[1].T)
            self.weights[0] = self.weights[0] - self.lr * 2 * np.dot(X_train.T, dz1)
            self.weights[1] = self.weights[1] - self.lr * 2 * (np.dot(self.weighted_sum_dict[0].T, (y_pred - y_train)))
            self.bias[0] = self.bias[0] - self.lr * 2 * np.sum(dz1, axis=0)
            self.bias[1] = self.bias[1] - self.lr * 2 * np.sum(y_pred - y_train, axis=0)


    def predict(self, X_test):

        return self.forward_probagation(X_test)

# Back probagation for Regression using Early Stopping

In [1]:
import numpy as np
from sklearn.metrics import mean_squared_error

class Backprobagation:
    def __init__(self, epochs=100, lr=0.001,patience_limit = 10):
        self.epochs = epochs
        self.lr = lr
        self.weighted_sum_dict = {}
        self.weights = {}
        self.bias = {}
        # initialization stage
        self.weights[0] = np.ones((2, 2))
        self.bias[0] = np.ones(2,)
        self.weights[1] = np.ones((2, 1))
        self.bias[1] = np.ones(1,)

        # Early stopping data
        self.val_x  = None
        self.val_y = None

        # Early stopping parameters
        self.best_val_loss = float('inf')
        self.patience_counter = 0
        self.patience_limit  = patience_limit
        self.best_weights = {}
        self.best_biases = {}
        self.early_stop = False


    def forward_probagation(self, X_train):
        # forward propagation
        self.weighted_sum_dict[0] = np.dot(X_train, self.weights[0]) + self.bias[0]
        self.weighted_sum_dict[1] = np.dot(self.weighted_sum_dict[0], self.weights[1]) + self.bias[1]
        y_pred = self.weighted_sum_dict[1]
        return y_pred


    def early_stopping(self,X_train,y_train):

      predictions = self.predict(self.val_x)
      curr_mse = mean_squared_error(self.val_y,predictions)
      if curr_mse < self.best_val_loss :
        self.best_val_loss = curr_mse
        # storing weights
        self.best_weights[0] = self.weights[0]
        self.best_weights[1] = self.weights[1]

        # storing biases
        self.best_biases[0] = self.bias[0]
        self.best_biases[1] = self.bias[1]


      elif curr_mse>=self.best_val_loss :
        if self.patience_counter != self.patience_limit:
          self.patience_counter+=1
        else:
          self.early_stop = True

      return




    def fit(self, X_train, y_train):
        X_train_,self.val_x,y_train_,self.val_y = train_test_split(X_train,y_train,random_state=42)
        for i in range(self.epochs):
            if self.early_stop == True:
              self.weights[0] = self.best_weights[0]
              self.weights[1] = self.best_weights[1]

              self.bias[0] = self.best_biases[0]
              self.bias[1] = self.best_biases[1]
              return

            self.early_stopping(X_train,y_train)

            y_pred = self.forward_probagation(X_train)
            loss = np.mean((y_train - y_pred) ** 2)
            dz1 = np.dot((y_pred - y_train), self.weights[1].T)
            self.weights[0] = self.weights[0] - self.lr * 2 * np.dot(X_train.T, dz1)
            self.weights[1] = self.weights[1] - self.lr * 2 * (np.dot(self.weighted_sum_dict[0].T, (y_pred - y_train)))
            self.bias[0] = self.bias[0] - self.lr * 2 * np.sum(dz1, axis=0)
            self.bias[1] = self.bias[1] - self.lr * 2 * np.sum(y_pred - y_train, axis=0)


    def predict(self, X_test):

        return self.forward_probagation(X_test)

# Back Probagation for Regression Using the Mini Batch Gradient Descent

In [ ]:
import numpy as np
import math
class Backprobagation:
    def __init__(self, epochs=100, lr=0.001):
        self.epochs = epochs
        self.lr = lr
        self.weighted_sum_dict = {}
        self.weights = {}
        self.bias = {}
        # initialization stage
        self.weights[0] = np.ones((2, 2))
        self.bias[0] = np.ones(2,)
        self.weights[1] = np.ones((2, 1))
        self.bias[1] = np.ones(1,)

    def forward_probagation(self, X_train):
        # forward propagation
        self.weighted_sum_dict[0] = np.dot(X_train, self.weights[0]) + self.bias[0]
        self.weighted_sum_dict[1] = np.dot(self.weighted_sum_dict[0], self.weights[1]) + self.bias[1]
        y_pred = self.weighted_sum_dict[1]
        return y_pred

    def fit(self, X_train, y_train,batch_size=2):
        rem = False
        if X_train.shape[0]%batch_size != 0:
          rem = True
        num_batches = math.ceil(X_train.shape[0]/batch_size)
        df = np.concatenate((X_train, y_train.reshape(-1, 1)), axis=1)
        for j in range(self.epochs):
          # shuffling data
          np.random.shuffle(df)
          for i in range(num_batches):
              # creating batches
              tmp_df = df.copy()
              if rem and i == num_batches-1:
                tmp_df = tmp_df[i*batch_size:]
              else:
                tmp_df = tmp_df[i*batch_size:(i+1)*batch_size]
              X_train = tmp_df[:,0:2]
              y_train = tmp_df[:,-1].reshape(-1, 1)
              # calculating gradient
              y_pred = self.forward_probagation(X_train)
              loss = np.mean((y_train - y_pred) ** 2)
              dz1 = np.dot((y_pred - y_train), self.weights[1].T)
              self.weights[0] = self.weights[0] - self.lr * 2 * np.dot(X_train.T, dz1)
              self.weights[1] = self.weights[1] - self.lr * 2 * (np.dot(self.weighted_sum_dict[0].T, (y_pred - y_train)))
              self.bias[0] = self.bias[0] - self.lr * 2 * np.sum(dz1, axis=0)
              self.bias[1] = self.bias[1] - self.lr * 2 * np.sum(y_pred - y_train, axis=0)


    def predict(self, X_test):

        return self.forward_probagation(X_test)

# Back probagation for classification

In [ ]:
import numpy as np

class Backprobagation:
    def __init__(self, epochs=100, lr=0.001):
        self.epochs = epochs
        self.lr = lr
        self.weighted_sum_dict = {}
        self.weights = {}
        self.bias = {}
        self.activation_dict = {}
        # initialization stage
        self.weights[0] = np.ones((2, 2))
        self.bias[0] = np.ones(2,)
        self.weights[1] = np.ones((2, 1))
        self.bias[1] = np.ones(1,)

    def forward_probagation(self, X_train):
        # forward propagation
        self.weighted_sum_dict[0] = np.dot(X_train, self.weights[0]) + self.bias[0]
        self.activation_dict[0] = 1 / (1 + np.exp(-self.weighted_sum_dict[0]))

        self.weighted_sum_dict[1] = np.dot(self.activation_dict[0], self.weights[1]) + self.bias[1]
        self.activation_dict[1] = 1 / (1 + np.exp(-self.weighted_sum_dict[1]))

        y_pred = self.activation_dict[1]
        return y_pred

    def fit(self, X_train, y_train):
        for i in range(self.epochs):
            y_pred = self.forward_probagation(X_train)
            loss = -(1/y_train.shape[0]) * ((y_train@np.log(y_pred)) + ((1-y_train)@np.log(1-y_pred)))



            dz1 = ((y_pred-y_train) @ self.weights[1].T) * (self.activation_dict[0] * (1-self.activation_dict[0]))


            self.weights[0] = self.weights[0] - self.lr *  np.dot(X_train.T, dz1)
            self.weights[1] = self.weights[1] - self.lr *  (np.dot(self.activation_dict[0].T, (y_pred - y_train)))


            self.bias[0] = self.bias[0] - self.lr *  np.sum(dz1, axis=0)
            self.bias[1] = self.bias[1] - self.lr *  np.sum(y_pred - y_train, axis=0)

    def predict(self, X_test):
      if hasattr(X_test, "values"):
          X_test = X_test.values
      pred = self.forward_probagation(X_test)
      return (pred>=0.5).astype(int).flatten()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
X = df.drop(columns='package')
y = df['package']
X_train,X_test,y_train,y_test = train_test_split(X,y)


In [ ]:
my_bp = Backprobagation(epochs=50000, lr=0.0001)
my_bp.fit(X_train.values, y_train.values.reshape(-1, 1))

In [ ]:
r2_score(y_test,my_bp.predict(X_test))

0.9880261788598822

# Tensor flow

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [ ]:
model = Sequential()
model.add(Dense(2,activation='linear',input_dim=2))
model.add(Dense(1,activation='linear'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 2)              │             6 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │             3 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9 (36.00 B)

 Trainable params: 9 (36.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(loss='mean_squared_error',optimizer='Adam')

In [ ]:
model.fit(X_train,y_train,epochs=500)

Epoch 1/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - loss: 45.9698
Epoch 2/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 144ms/step - loss: 45.4955
Epoch 3/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step - loss: 45.0249
Epoch 4/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step - loss: 44.5581
Epoch 5/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - loss: 44.0950
Epoch 6/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - loss: 43.6357
Epoch 7/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step - loss: 43.1800
Epoch 8/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - loss: 42.7280
Epoch 9/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - loss: 42.2796
Epoch 10/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - loss: 41.8349
Epoch 11/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - loss: 41.3939
Epoch 12/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - loss: 40.9564
Epoch 13/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - loss: 40.5226
Epoch 14/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - loss: 40.0923
Epoch 15/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - loss: 39.6656
Epo

In [ ]:
r2_score(y_test,model.predict(X_test))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step


0.18896504465182073